# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the FAIR^2 dataset using the `mlcroissant` library, referencing dataset entities by their `@id` as required for working with Croissant schemas.

### Dataset Source
The dataset schema is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load dataset metadata and initialize Dataset object
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Published on: {metadata.datePublished}")
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their unique `@id`s.

### List Record Sets
Each tabular data collection is represented as a `RecordSet` with its own `@id`. We enumerate all record sets and their available fields and columns.

In [ ]:
# List all available record sets and their associated fields and columns
record_sets = dataset.metadata.recordSet
if record_sets is None or len(record_sets) == 0:
    print("No record sets found in this schema. Please check your version or schema definition.")
else:
    print("Available RecordSets:")
    for rset in record_sets:
        print(f"RecordSet @id: {rset['@id']}")
        # List fields and columns
        fields = rset.get('field', [])
        columns = rset.get('column', [])
        print(f"  Fields:")
        for field in fields:
            print(f"    Field @id: {field['@id']} Name: {field.get('name')} DataType: {field.get('dataType')}")
        print(f"  Columns:")
        for column in columns:
            print(f"    Column @id: {column['@id']} Name: {column.get('name')} DataType: {column.get('dataType')}")
        print()

## 3. Data Extraction
Load data from specific record sets into pandas DataFrames for analysis. Always reference a record set and field by their `@id`.

In [ ]:
# Identify all record set @id's for extraction
record_set_ids = [rset['@id'] for rset in dataset.metadata.recordSet] if dataset.metadata.recordSet else []

dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"DataFrame loaded for RecordSet @id: {rs_id}")
    print(f"Columns: {df.columns.tolist()}")
    print(df.head(), '\n')

# Choose the first record set for further exploration
if record_set_ids:
    test_record_set_id = record_set_ids[0]
    print(f"Sample data from RecordSet @id: {test_record_set_id}")
    print(dataframes[test_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping by key attributes.
All field references use column `@id` values.

In [ ]:
# EDA: Choose a numeric column and a group column from the first record set
df = dataframes[test_record_set_id] if record_set_ids else None
if df is not None:
    # Discover numeric columns using their @id
    # For demonstration, let's select the first numeric column
    numeric_columns_ids = [col for col in df.columns if df[col].dtype in ['int64', 'float64']]
    group_candidates_ids = [col for col in df.columns if df[col].dtype == 'object']
    if numeric_columns_ids:
        numeric_field_id = numeric_columns_ids[0]
        print(f"Using numeric field for filtering and normalization: {numeric_field_id}")
        # Set a threshold for filtering
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].notnull().sum()>0 else 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"].copy()].head())

        # Try grouping by another field
        if group_candidates_ids:
            group_field_id = group_candidates_ids[0]
            if group_field_id in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
                print(f"Grouped data by {group_field_id}:")
                print(grouped_df.head())
    else:
        print("No numeric fields available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields using matplotlib.
All axis field references use column `@id` values.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_columns_ids:
    # Histogram of selected numeric field
    plt.figure(figsize=(6,4))
    sns.histplot(df[numeric_field_id], bins=10, kde=True)
    plt.title(f'Histogram of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # If there are at least two numeric fields, plot scatter
    if len(numeric_columns_ids) > 1:
        plt.figure(figsize=(6,4))
        sns.scatterplot(x=df[numeric_columns_ids[0]], y=df[numeric_columns_ids[1]])
        plt.title(f'Scatter plot: {numeric_columns_ids[0]} vs. {numeric_columns_ids[1]}')
        plt.xlabel(numeric_columns_ids[0])
        plt.ylabel(numeric_columns_ids[1])
        plt.show()

## 6. Conclusion
This notebook explored the FAIR^2 dataset using the `mlcroissant` library, referencing all elements by their `@id` according to the Croissant schema. We:
- Loaded dataset metadata and displayed descriptive information
- Enumerated available record sets and their field and column `@id`s
- Loaded records from each record set into DataFrames
- Performed filtering and normalization on numeric fields using their `@id`
- Grouped, summarized, and visualized data distribution

**Key insights**: The dataset is structured for clinical and genetic analyses and contains rich metadata including biases, collection protocol, and limitations. The sample data is small and not generalizable, but is valuable for academic and clinical illustration.

For further analysis, consider integrating domain-specific transformations and cross-record set analysis, always referencing fields and entities by Croissant `@id`.